In [1]:
import requests
import pandas as pd
import re
import time
from bs4 import BeautifulSoup
import numpy as np
from tqdm import tqdm

In [2]:
url = "https://www.basketball-reference.com/teams/"
headers= {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers, timeout=20)
print(response.status_code)
soup = BeautifulSoup(response.text, "html.parser")

200


In [3]:
team_ids = ['ATL','BOS','NJN','CHA','CHI','CLE','DAL','DEN','DET','GSW','HOU','IND','LAC','LAL','MEM',
 'MIA','MIL','MIN','NOH','NYK','OKC','ORL','PHI','PHO','POR','SAC','SAS','TOR','UTA','WAS']

In [4]:
def get_team_basic_totals(team_id, team_name):

    url = f"https://www.basketball-reference.com/teams/{team_id}/stats_basic_totals.html"

    response = requests.get(url, headers=headers, timeout=20)

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    table = soup.find("table", id="stats")

    if table is None:
        print("Table not found:", team_id)
        return []

    rows = table.find("tbody").find_all("tr")[:7]

    column_names = {
        "season": "season",
        "lg_id": "league",
        "team_id": "team_in_season",
        "wins": "wins",
        "losses": "losses",
        "rank_team": "finish",
        "avg_age": "average_age",
        "avg_ht": "average_height",
        "avg_wt": "average_weight",
        "g": "games",
        "mp": "minutes",
        "fg": "field_goals",
        "fga": "field_goal_attempts",
        "fg_pct": "field_goal_percentage",
        "fg3": "three_points",
        "fg3a": "three_point_attempts",
        "fg3_pct": "three_point_percentage",
        "fg2": "two_points",
        "fg2a": "two_point_attempts",
        "fg2_pct": "two_point_percentage",
        "ft": "free_throws",
        "fta": "free_throw_attempts",
        "ft_pct": "free_throw_percentage",
        "orb": "offensive_rebounds",
        "drb": "defensive_rebounds",
        "trb": "total_rebounds",
        "ast": "assists",
        "stl": "steals",
        "blk": "blocks",
        "tov": "turnovers",
        "pf": "personal_fouls",
        "pts": "points"
    }
    
    rows = rows[:7]
    team_rows = []

    for row in rows:

        record = {"team_name": team_name, "team_id": team_id}

        
        for new_name in column_names.values():
            record[new_name] = np.nan

        cells = row.find_all(["th", "td"])

        for cell in cells:

            data_stat = cell.get("data-stat")

            if data_stat in column_names:

                value = cell.get_text( " ", strip=True)

                if value != "":
                    record[column_names[data_stat]] = value

        team_rows.append(record)

    return team_rows

In [5]:
all_basic_totals = []

for team_id in tqdm(team_ids):

    team_data = get_team_basic_totals(team_id, team_id)

    all_basic_totals.extend(team_data)

    time.sleep(5)

100%|██████████| 30/30 [02:56<00:00,  5.89s/it]


In [6]:
team_basic_totals = pd.DataFrame(all_basic_totals)

team_basic_totals

,team_name,team_id,season,league,team_in_season,wins,losses,finish,average_age,average_height,...,free_throw_percentage,offensive_rebounds,defensive_rebounds,total_rebounds,assists,steals,blocks,turnovers,personal_fouls,points
0,ATL,ATL,2025-26,NBA,ATL,46,36,1,25.2,6-7,...,.774,900,2670,3570,2471,768,384,1162,1612,9714
1,ATL,ATL,2024-25,NBA,ATL,40,42,2,24.9,6-6,...,.775,974,2675,3649,2426,798,419,1273,1564,9691
2,ATL,ATL,2023-24,NBA,ATL,36,46,3,26.2,6-6,...,.797,1024,2639,3663,2180,615,369,1110,1522,9703
3,ATL,ATL,2022-23,NBA,ATL,41,41,2,24.9,6-6,...,.818,920,2719,3639,2049,580,401,1060,1541,9711
4,ATL,ATL,2021-22,NBA,ATL,43,39,2,26.1,6-6,...,.812,823,2783,3606,2017,587,348,972,1534,9343
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
205,WAS,WAS,2023-24,NBA,WAS,15,67,5,24.9,6-6,...,.764,755,2613,3368,2288,623,415,1147,1637,9327
206,WAS,WAS,2022-23,NBA,WAS,35,47,3,26.2,6-7,...,.785,774,2804,3578,2083,561,424,1158,1539,9279
207,WAS,WAS,2021-22,NBA,WAS,35,47,4,25.9,6-6,...,.783,737,2798,3535,2052,522,406,1077,1545,8907
208,WAS,WAS,2020-21,NBA,WAS,34,38,3,26.6,6-6,...,.769,697,2557,3254,1835,528,297,1037,1555,8398


In [8]:
team_basic_totals.to_csv("team_basic_totals.csv", index=False, encoding="utf-8")